In [ ]:
# ============================================================
#
# Цель:
# 1) Получить baseline (LogisticRegression)
# 2) Проверить бустинг (LightGBM) на сабсемпле
# 3) Обучить лучшую модель на полном датасете
# 4) Попробовать CatBoost с корректной обработкой категорий
#
# Данные:
# dataset_fe_AB.pkl — результат Feature Engineering (A + B)
# ============================================================

In [1]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    StratifiedShuffleSplit,
)
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool

In [2]:
# =========================
# Константы проекта
# =========================
RANDOM_STATE = 42
SUB_FRAC = 0.05   # 5% для быстрых экспериментов

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_fe_AB.pkl"

In [4]:
# ============================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================

def load_data(path: Path) -> pd.DataFrame:
    """Загрузка датасета"""
    df = pd.read_pickle(path)
    assert "flag" in df.columns
    return df


def make_train_test(df: pd.DataFrame):
    """Стандартный train/test split (80/20, stratified)"""
    X = df.drop(columns=["flag"])
    y = df["flag"]

    return train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_STATE,
    )


def make_subsample(df: pd.DataFrame, frac: float):
    """Стратифицированный сабсемпл"""
    sss = StratifiedShuffleSplit(
        n_splits=1,
        test_size=frac,
        random_state=RANDOM_STATE,
    )
    for _, idx in sss.split(df, df["flag"]):
        return df.iloc[idx].copy()


def evaluate_auc(model, X_train, y_train, X_test, y_test):
    """Обучение модели и расчёт ROC-AUC"""
    model.fit(X_train, y_train)
    return roc_auc_score(
        y_test,
        model.predict_proba(X_test)[:, 1]
    )

In [5]:
# ============================================================
# ШАГ 1. ЗАГРУЗКА ДАННЫХ
# ============================================================
df = load_data(DATA_PATH)
print("Full dataset shape:", df.shape)

Full dataset shape: (3000000, 73)


In [6]:
# ============================================================
# ШАГ 1.1 Получение сабсемпла и разбиение на тестовые и тренировочные данные
# ============================================================
df_small = make_subsample(df, SUB_FRAC)
X_train, X_test, y_train, y_test = make_train_test(df_small)

In [ ]:
# ============================================================
# ШАГ 2. BASELINE — LOGISTIC REGRESSION (SANITY CHECK)
# ============================================================


logreg = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

roc_auc_lr = evaluate_auc(
    logreg, X_train, y_train, X_test, y_test
)

print("ROC-AUC LogisticRegression:", roc_auc_lr)

In [ ]:
# ============================================================
# ШАГ 2.1. ДЕРЕВЬЯ И СЛУЧАЙНЫЙ ЛЕС (BASELINES)
# Использовались для сравнения. В финал не прошли.
# ============================================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

dt = DecisionTreeClassifier(
    max_depth=6,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

roc_auc_dt = evaluate_auc(dt, X_train, y_train, X_test, y_test)
roc_auc_rf = evaluate_auc(rf, X_train, y_train, X_test, y_test)

print("ROC-AUC DecisionTree:", roc_auc_dt)
print("ROC-AUC RandomForest:", roc_auc_rf)


In [ ]:
# ============================================================
# ШАГ 3. LIGHTGBM НА САБСЕМПЛЕ (ОСНОВНАЯ МОДЕЛЬ)
# ============================================================
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    min_child_samples=100,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

roc_auc_lgbm = evaluate_auc(
    lgbm, X_train, y_train, X_test, y_test
)

print("ROC-AUC LightGBM (subsample):", roc_auc_lgbm)

### Тюнинг LightGBM (Feature Engineering A + B)

| run | num_leaves | min_child_samples | learning_rate | n_estimators | imbalance_handling | ROC-AUC |
|-----|------------|-------------------|---------------|--------------|--------------------|---------|
| base| 64         | 100               | 0.05          | 300          | class_weight       | 0.71316 |
| 1   | 128        | 100               | 0.05          | 300          | class_weight       | 0.70227 |
| 2   | 32         | 100               | 0.05          | 300          | class_weight       | 0.70854 |
| 3   | 64         | 50                | 0.05          | 300          | class_weight       | 0.70276 |
| 4   | 64         | 200               | 0.05          | 300          | class_weight       | 0.70810 |
| 5   | 64         | 100               | 0.03          | 600          | class_weight       | 0.70606 |
| 6   | 64         | 100               | 0.07          | 300          | class_weight       | 0.69746 |
| 7   | 64         | 100               | 0.05          | 300          | scale_pos_weight   | 0.70692 |
| 8   | 128        | 50                | 0.05          | 300          | class_weight       | 0.69391 |


**Вывод:**
Базовая конфигурация LightGBM (num_leaves=64, min_child_samples=100, learning_rate=0.05)
показала наилучшее качество. Увеличение сложности модели, изменение скорости обучения
и альтернативные способы учета дисбаланса классов не привели к улучшению ROC-AUC.



После выбора оптимальной конфигурации модель была обучена на полном датасете,
что позволило увеличить ROC-AUC с ~0.71 до 0.7255.


In [6]:
# ============================================================
# ШАГ 4. LIGHTGBM НА ПОЛНОМ ДАТАСЕТЕ (ФИНАЛ)
# ============================================================
X_train_full, X_test_full, y_train_full, y_test_full = make_train_test(df)

lgbm_final = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    min_child_samples=100,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

lgbm_final.fit(X_train_full, y_train_full)

roc_auc_full = roc_auc_score(
    y_test_full,
    lgbm_final.predict_proba(X_test_full)[:, 1]
)

print("ROC-AUC LightGBM (full data):", roc_auc_full)

[LightGBM] [Info] Number of positive: 85154, number of negative: 2314846
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.733094 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13316
[LightGBM] [Info] Number of data points in the train set: 2400000, number of used features: 67
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
ROC-AUC LightGBM (full data): 0.7255092690247111


In [7]:
# ============================================================
# ШАГ 5. CATBOOST — ПРОВЕРКА ГИПОТЕЗЫ О КАТЕГОРИЯХ
# ============================================================
# В CatBoost cat_features могут быть ТОЛЬКО int / str.
# Поэтому отбираем категории по dtype, а не по имени.

numeric_fe_cols = {
    "outstanding_to_limit",
    "overdue_to_limit",
    "maxoverdue_to_limit",
    "share_long_overdue",
}

cat_cols = [
    c for c in X_train.columns
    if (c not in numeric_fe_cols)
    and (X_train[c].dtype.kind in {"i", "O"})
]

# Защита: в cat_cols не должно быть float
assert not any(X_train[c].dtype.kind == "f" for c in cat_cols)

cat_idx = [X_train.columns.get_loc(c) for c in cat_cols]

train_pool = Pool(
    X_train,
    y_train,
    cat_features=cat_idx
)

test_pool = Pool(
    X_test,
    y_test,
    cat_features=cat_idx
)

cb = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    early_stopping_rounds=200,
    verbose=200,
    task_type="CPU",
)

cb.fit(train_pool, eval_set=test_pool, use_best_model=True)

roc_auc_cb = roc_auc_score(
    y_test,
    cb.predict_proba(X_test)[:, 1]
)

print("ROC-AUC CatBoost (subsample):", roc_auc_cb)

0:	test: 0.6630890	best: 0.6630890 (0)	total: 562ms	remaining: 28m 4s
200:	test: 0.7045245	best: 0.7124756 (87)	total: 1m 11s	remaining: 16m 40s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.7124755716
bestIteration = 87

Shrink model to first 88 iterations.
ROC-AUC CatBoost (subsample): 0.7124755715751225


In [8]:
import joblib
joblib.dump(lgbm_final, "../artifacts/model.pkl")

['../artifacts/model.pkl']